# S8 — AndinaLog 03B | Segmentación exploratoria de viajes

K-Means se ajusta únicamente con viajes de la partición AJUSTE. Los objetivos futuros se excluyen de la formación y se usan después para describir los grupos.

## 1. Datos de entrenamiento

Se conserva el split temporal de S6–S7. La unidad de clustering es el viaje, evitando que un trayecto con más lecturas tenga más peso.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ENTORNO = "auto"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
SEMILLA = 42
TARGET_CLASIFICACION = "clasificacion_objetivo_60min"
TARGET_REGRESION = "max_desvio_termico_proximos_60min_c"

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
    else:
        raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "proyecto-integrador/03_EDA/salidas_v4_eventos/andinalog_03b_evidencia_3_lecturas_con_eventos.csv").is_file()), None)
    if raiz is None: raise FileNotFoundError("No se encontró la raíz del proyecto")
    return raiz

RAIZ = encontrar_raiz()
RUTA_DATOS = RAIZ / "proyecto-integrador/03_EDA/salidas_v4_eventos/andinalog_03b_evidencia_3_lecturas_con_eventos.csv"
RUTA_SPLIT = RAIZ / "proyecto-integrador/04_regresion/salidas_s6/asignacion_split_viajes.csv"
SALIDAS = RAIZ / "proyecto-integrador/06_clustering/salidas_s8"
df = pd.read_csv(RUTA_DATOS, encoding="utf-8-sig")
split = pd.read_csv(RUTA_SPLIT, encoding="utf-8-sig")
df["timestamp_bolivia"] = pd.to_datetime(df["timestamp_bolivia"], errors="raise")
df["particion"] = df["viaje_id"].map(split.set_index("viaje_id")["particion"])
train = df.loc[df["particion"].eq("AJUSTE")].copy()
assert train["viaje_id"].nunique() == 796
print("Lecturas de ajuste:", len(train), "| viajes:", train["viaje_id"].nunique())


Lecturas de ajuste: 19017 | viajes: 796


## 2. Agregación por viaje y control de fuga

Se resumen condiciones térmicas, humedad, estabilidad, capacidad, pedido y eventos históricos. Categorías y objetivos no forman parte de la distancia.

In [2]:
# Una fila por viaje. Los objetivos se agregan solo para perfilar después, no para formar clusters.
train["pendiente_abs"] = train["pendiente_c_por_min"].abs()
viajes = train.groupby("viaje_id", as_index=False).agg(
    camion_id=("camion_id_tratado", "first"), producto_id=("producto_id", "first"),
    categoria_logistica=("categoria_logistica_tratada", "first"), tipo_camion=("tipo_camion_tratado", "first"),
    centro_distribucion=("centro_distribucion_tratado", "first"), lecturas=("fila_bronze", "size"),
    duracion_min=("minutos_desde_inicio", "max"), temp_media_c=("temp_c", "mean"),
    temp_std_c=("temp_c", "std"), temp_min_c=("temp_c", "min"), temp_max_c=("temp_c", "max"),
    humedad_media_pct=("humedad_pct", "mean"), humedad_std_pct=("humedad_pct", "std"),
    desvio_actual_medio_c=("desvio_respecto_umbral_c", "mean"),
    desvio_actual_max_c=("desvio_respecto_umbral_c", "max"),
    pendiente_abs_media=("pendiente_abs", "mean"), pendiente_abs_max=("pendiente_abs", "max"),
    capacidad_kg=("capacidad_kg_tratada", "first"), cantidad_solicitada=("cantidad_solicitada_tratado", "first"),
    entrega_prometida_hrs=("tiempo_entrega_prometido_hrs_tratado", "first"),
    eventos_previos_24h_max=("eventos_previos_24h", "max"),
    alertas_temp_previas_24h_max=("alertas_temp_previas_24h", "max"),
    fallas_motor_previas_24h_max=("fallas_motor_previas_24h", "max"),
    eventos_alta_previos_24h_max=("eventos_alta_previos_24h", "max"),
    lecturas_con_evento_previo_pct=("tiene_evento_previo_24h", "mean"),
    tuvo_desviacion_futura=(TARGET_CLASIFICACION, "max"),
    tasa_lecturas_desviacion_futura=(TARGET_CLASIFICACION, "mean"),
    max_desvio_futuro_c=(TARGET_REGRESION, "max"),
)
FEATURES_CLUSTER = [
    "duracion_min", "temp_media_c", "temp_std_c", "temp_min_c", "temp_max_c",
    "humedad_media_pct", "humedad_std_pct", "desvio_actual_medio_c", "desvio_actual_max_c",
    "pendiente_abs_media", "pendiente_abs_max", "capacidad_kg", "cantidad_solicitada",
    "entrega_prometida_hrs", "eventos_previos_24h_max", "alertas_temp_previas_24h_max",
    "fallas_motor_previas_24h_max", "eventos_alta_previos_24h_max", "lecturas_con_evento_previo_pct",
]
OBJETIVOS = {"tuvo_desviacion_futura", "tasa_lecturas_desviacion_futura", "max_desvio_futuro_c",
             TARGET_CLASIFICACION, TARGET_REGRESION}
assert not set(FEATURES_CLUSTER) & OBJETIVOS
assert len(viajes) == 796 and viajes["viaje_id"].is_unique
print("Variables de clustering:", len(FEATURES_CLUSTER))


Variables de clustering: 19


## 3. Preparación y selección de k

Los faltantes se imputan con la mediana y las variables se estandarizan. Se comparan k=2 a k=6 mediante inercia, silhouette y tamaño de los grupos.

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

preparacion = Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())])
X = preparacion.fit_transform(viajes[FEATURES_CLUSTER])
resultados_k, etiquetas_por_k = [], {}
for k in range(2, 7):
    modelo = KMeans(n_clusters=k, random_state=SEMILLA, n_init=20)
    etiquetas = modelo.fit_predict(X)
    etiquetas_por_k[k] = etiquetas
    resultados_k.append({"k": k, "inercia": modelo.inertia_, "silhouette": silhouette_score(X, etiquetas),
                         "cluster_min": int(pd.Series(etiquetas).value_counts().min()),
                         "cluster_max": int(pd.Series(etiquetas).value_counts().max())})
tabla_k = pd.DataFrame(resultados_k)
K_MEJOR = int(tabla_k.sort_values(["silhouette", "k"], ascending=[False, True]).iloc[0]["k"])
K_ALTERNATIVO = 3
viajes["cluster_mejor"] = etiquetas_por_k[K_MEJOR]
viajes["cluster_k3"] = etiquetas_por_k[K_ALTERNATIVO]
print(tabla_k.round(4).to_string(index=False))
print("Mejor k por silhouette:", K_MEJOR)


 k    inercia  silhouette  cluster_min  cluster_max
 2 10808.9411      0.3032          386          410
 3  9360.1203      0.3310           78          409
 4  8448.5598      0.3209           76          322
 5  7820.2889      0.3198           34          321
 6  7280.1929      0.2766           34          322
Mejor k por silhouette: 3


## 4. Perfil posterior

Después de formar los grupos se describen las desviaciones futuras. Esto ayuda a interpretar, pero el objetivo no influyó en K-Means.

In [4]:
def perfilar(col_cluster):
    return viajes.groupby(col_cluster).agg(
        viajes=("viaje_id", "size"),
        pct_viajes_con_desviacion=("tuvo_desviacion_futura", lambda s: 100*s.mean()),
        tasa_lecturas_desviacion_pct=("tasa_lecturas_desviacion_futura", lambda s: 100*s.mean()),
        max_desvio_futuro_medio_c=("max_desvio_futuro_c", "mean"),
        temp_media_c=("temp_media_c", "mean"), temp_max_c=("temp_max_c", "mean"),
        desvio_actual_max_c=("desvio_actual_max_c", "mean"), humedad_media_pct=("humedad_media_pct", "mean"),
        pendiente_abs_max=("pendiente_abs_max", "mean"), duracion_min=("duracion_min", "mean"),
        capacidad_kg=("capacidad_kg", "mean"), cantidad_solicitada=("cantidad_solicitada", "mean"),
        eventos_previos_24h_max=("eventos_previos_24h_max", "mean"),
        alertas_temp_previas_24h_max=("alertas_temp_previas_24h_max", "mean"),
    ).reset_index()

perfil_mejor = perfilar("cluster_mejor")
perfil_k3 = perfilar("cluster_k3")
print("Perfil mejor k")
print(perfil_mejor.round(3).to_string(index=False))
print("Perfil k=3")
print(perfil_k3.round(3).to_string(index=False))


Perfil mejor k
 cluster_mejor  viajes  pct_viajes_con_desviacion  tasa_lecturas_desviacion_pct  max_desvio_futuro_medio_c  temp_media_c  temp_max_c  desvio_actual_max_c  humedad_media_pct  pendiente_abs_max  duracion_min  capacidad_kg  cantidad_solicitada  eventos_previos_24h_max  alertas_temp_previas_24h_max
             0      78                    100.000                        26.510                      5.493        -1.743       4.404                5.493             74.791              0.239       689.231      5782.051              268.103                    0.577                         0.141
             1     309                     23.625                         2.111                     -0.214        -3.978      -2.433               -0.204             75.095              0.082       689.126      5255.663              248.153                    0.256                         0.065
             2     409                     18.093                         2.326                  

## 5. Evidencias

Se conservan la asignación de los 796 viajes, los perfiles de la solución con mejor silhouette y la alternativa k=3.

In [5]:
SALIDAS.mkdir(parents=True, exist_ok=True)
tabla_k.to_csv(SALIDAS / "evaluacion_k_s8.csv", index=False, encoding="utf-8-sig")
viajes.to_csv(SALIDAS / "asignacion_clusters_viajes_ajuste_s8.csv", index=False, encoding="utf-8-sig")
perfil_mejor.to_csv(SALIDAS / "perfil_clusters_mejor_k_s8.csv", index=False, encoding="utf-8-sig")
perfil_k3.to_csv(SALIDAS / "perfil_clusters_k3_s8.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(tabla_k["k"], tabla_k["inercia"], marker="o")
axes[0].set(title="Inercia por número de clusters", xlabel="k", ylabel="Inercia")
axes[1].plot(tabla_k["k"], tabla_k["silhouette"], marker="o", color="darkorange")
axes[1].axvline(K_MEJOR, linestyle="--", color="gray", label=f"Mejor k={K_MEJOR}")
axes[1].set(title="Silhouette por número de clusters", xlabel="k", ylabel="Silhouette")
axes[1].legend()
plt.tight_layout()
plt.savefig(SALIDAS / "seleccion_k_s8.png", dpi=150, bbox_inches="tight")
plt.close()
print("Salidas guardadas en", SALIDAS)


Salidas guardadas en c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\06_clustering\salidas_s8


## 6. Alcance

Los clusters son perfiles exploratorios del conjunto de ajuste. No son categorías oficiales, causas ni etiquetas para producción hasta validarlos con el negocio y con periodos futuros.